# Checkpoint-1200 On-Policy RL Train

Continue from `checkpoint-1200` with reward-driven group-relative policy-gradient training plus multitask SFT replay.


In [ ]:
# 1) Install dependencies, then restart runtime once.
# After restart, run this cell again and continue.
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_pilot_c_deps_installed")

if Path("/content").exists() and not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "huggingface_hub",
        "hf_xet",
        "modelscope",
        "jedi",
        "pandas==2.2.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
elif Path("/content").exists():
    print("Dependencies already installed. Continue.")
else:
    print("Local environment detected. Skipping Colab dependency install.")


In [ ]:
# 2) Setup: Drive, data, model cache, run paths
from google.colab import drive
from pathlib import Path
import os
import shutil

drive_root = Path("/content/drive")
if drive_root.exists() and not os.path.ismount(str(drive_root)) and any(drive_root.iterdir()):
    print("Removing local pre-mount /content/drive contents:", sorted(str(p) for p in drive_root.iterdir())[:20])
    shutil.rmtree(drive_root)
drive_root.mkdir(parents=True, exist_ok=True)
drive.mount("/content/drive")

import ast
import copy
import gc
import glob
import itertools
import json
import math
import random
import re
import subprocess
import zipfile
from datetime import datetime

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, TrainerCallback, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()

SNU_ROOT = Path("/content/drive/MyDrive/SNU_AI_Challenge")
assert SNU_ROOT.exists(), SNU_ROOT

ZIP_PATH = SNU_ROOT / "snuaichallenge.zip"
DATA_DIR = Path("/content/snuaichallenge_data")
TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
TRAIN_IMAGE_DIR = DATA_DIR / "train"
TEST_IMAGE_DIR = DATA_DIR / "test"

if not TRAIN_CSV.exists() or not TRAIN_IMAGE_DIR.is_dir():
    print("Extracting:", ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

assert TRAIN_CSV.exists(), TRAIN_CSV
assert TEST_CSV.exists(), TEST_CSV
assert TRAIN_IMAGE_DIR.is_dir(), TRAIN_IMAGE_DIR
assert TEST_IMAGE_DIR.is_dir(), TEST_IMAGE_DIR

MODEL_REPO_ID = "Qwen/Qwen2-VL-7B-Instruct"
USE_MODELSCOPE_BASE_MODEL = False
DRIVE_MODEL_DIR = SNU_ROOT / "model_cache/Qwen2-VL-7B-Instruct"
LOCAL_MODEL_DIR = Path("/content/Qwen2-VL-7B-Instruct")


def print_runtime_storage():
    print("Storage check for /content:")
    try:
        subprocess.run(["df", "-h", "/content"], check=False)
    except Exception as exc:
        total, used, free = shutil.disk_usage("/content")
        print(f"/content free: {free / (1024 ** 3):.1f} GB / total: {total / (1024 ** 3):.1f} GB ({exc})")

    try:
        meminfo = {}
        with open("/proc/meminfo", "r", encoding="utf-8") as handle:
            for line in handle:
                key, value = line.split(":", 1)
                meminfo[key] = int(value.strip().split()[0]) / (1024 ** 2)
        print(f"RAM available: {meminfo.get('MemAvailable', 0):.1f} GB / total: {meminfo.get('MemTotal', 0):.1f} GB")
    except Exception as exc:
        print("RAM check skipped:", exc)

    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f"GPU memory free: {free / (1024 ** 3):.1f} GB / total: {total / (1024 ** 3):.1f} GB")


def model_cache_is_complete(model_dir):
    model_dir = Path(model_dir)
    if not (model_dir / "config.json").exists():
        return False
    has_weight = (model_dir / "model.safetensors.index.json").exists() or bool(list(model_dir.glob("*.safetensors")))
    has_processor = any((model_dir / name).exists() for name in ["preprocessor_config.json", "processor_config.json", "tokenizer.json", "tokenizer_config.json"])
    return bool(has_weight and has_processor)


def copy_drive_cache_to_local():
    if not model_cache_is_complete(DRIVE_MODEL_DIR):
        raise FileNotFoundError(f"Drive model cache is incomplete or missing: {DRIVE_MODEL_DIR}")

    if model_cache_is_complete(LOCAL_MODEL_DIR):
        print("Using existing local model:", LOCAL_MODEL_DIR)
        return

    print("Copying base model from Drive to Colab local disk...")
    print("  from:", DRIVE_MODEL_DIR)
    print("  to  :", LOCAL_MODEL_DIR)
    if LOCAL_MODEL_DIR.exists():
        shutil.rmtree(LOCAL_MODEL_DIR)
    shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)
    assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR


def download_base_model_to_local_and_cache():
    print("Base model cache not found. Downloading with Hugging Face snapshot_download to local disk first:")
    print("repo:", MODEL_REPO_ID)
    if LOCAL_MODEL_DIR.exists() and not model_cache_is_complete(LOCAL_MODEL_DIR):
        shutil.rmtree(LOCAL_MODEL_DIR)
    from huggingface_hub import snapshot_download
    hf_token = os.environ.get("HF_TOKEN")
    model_dir = snapshot_download(
        repo_id=MODEL_REPO_ID,
        local_dir=str(LOCAL_MODEL_DIR),
        token=hf_token,
        max_workers=8,
    )
    print("Downloaded:", model_dir)
    assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR

    DRIVE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(DRIVE_MODEL_DIR) + ".tmp")
    if tmp.exists():
        shutil.rmtree(tmp)
    shutil.copytree(LOCAL_MODEL_DIR, tmp)
    if DRIVE_MODEL_DIR.exists():
        shutil.rmtree(DRIVE_MODEL_DIR)
    os.replace(tmp, DRIVE_MODEL_DIR)
    print("Saved to Drive:", DRIVE_MODEL_DIR)


def ensure_base_model_path():
    print_runtime_storage()

    if model_cache_is_complete(LOCAL_MODEL_DIR):
        print("Using local model:", LOCAL_MODEL_DIR)
    elif model_cache_is_complete(DRIVE_MODEL_DIR):
        copy_drive_cache_to_local()
    elif not USE_MODELSCOPE_BASE_MODEL:
        download_base_model_to_local_and_cache()
    else:
        print("Base model cache not found. Downloading via ModelScope:", MODEL_REPO_ID)
        from modelscope import snapshot_download as modelscope_snapshot_download
        model_dir = modelscope_snapshot_download(MODEL_REPO_ID, cache_dir="/content/modelscope_cache")
        if LOCAL_MODEL_DIR.exists():
            shutil.rmtree(LOCAL_MODEL_DIR)
        shutil.copytree(model_dir, LOCAL_MODEL_DIR)
        assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR
        DRIVE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
        tmp = Path(str(DRIVE_MODEL_DIR) + ".tmp")
        if tmp.exists():
            shutil.rmtree(tmp)
        shutil.copytree(LOCAL_MODEL_DIR, tmp)
        if DRIVE_MODEL_DIR.exists():
            shutil.rmtree(DRIVE_MODEL_DIR)
        os.replace(tmp, DRIVE_MODEL_DIR)
        print("Saved to Drive:", DRIVE_MODEL_DIR)

    print_runtime_storage()
    print("Using local model:", LOCAL_MODEL_DIR)
    return str(LOCAL_MODEL_DIR)


MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = True

OUTPUT_ROOT = SNU_ROOT / "qwen2vl_7b_multitask_bipair_conditional_v1"
BASE_RUN_ID = "20260721_011312"
BASE_CHECKPOINT_NAME = "checkpoint-1200"
BASE_RUN_ROOT = OUTPUT_ROOT / "runs" / BASE_RUN_ID
BASE_OUTPUT_DIR = BASE_RUN_ROOT / "multitask_bipair_conditional"
BASE_CHECKPOINT_DIR = BASE_OUTPUT_DIR / BASE_CHECKPOINT_NAME
assert BASE_CHECKPOINT_DIR.is_dir(), BASE_CHECKPOINT_DIR

RL_RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_rl_from_checkpoint1200"
RUN_ID = RL_RUN_ID
RUN_ROOT = OUTPUT_ROOT / "runs" / RUN_ID
OUTPUT_DIR = RUN_ROOT / "onpolicy_rl_from_checkpoint1200"
EVAL_DIR = OUTPUT_DIR / "eval"
CHECKPOINT_ROOT = OUTPUT_DIR / "checkpoints"
for path in [RUN_ROOT, OUTPUT_DIR, EVAL_DIR, CHECKPOINT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
VALID_RATIO = 0.10
TRAIN_ROWS = None
VALID_ROWS = None
QUICK_EVAL_ROWS = 50
RUN_QUICK_EVAL_DURING_TRAIN = True

TASK_RATIOS = {
    "order": 0.30,
    "pairwise": 0.25,
    "first": 0.10,
    "last": 0.10,
    "fixed_first": 0.10,
    "fixed_last": 0.10,
    "fixed_endpoints": 0.05,
}
TASK_LOSS_WEIGHTS = {task: 1.0 for task in TASK_RATIOS}

LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

LEARNING_RATE = 5e-7
# Safety default: run a single sanity step first. After checking logs, raise to 10, then 200.
RL_STEPS = 1
ROLLOUTS_PER_SAMPLE = 6
TEMPERATURE = 0.8
SFT_REPLAY_WEIGHT = 0.20
MAX_GRAD_NORM = 1.0
SAVE_STEPS = 1
EVAL_STEPS = 50
LOGGING_STEPS = 1
HARD_SAMPLE_RATIO = 0.0
RANDOM_SAMPLE_RATIO = 1.0
SFT_REPLAY_BATCH_SIZE = 1
RL_CANDIDATE_BATCH_SIZE = 1
RL_GRAD_CANDIDATE_BATCH_SIZE = 1

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("base checkpoint:", BASE_CHECKPOINT_DIR)
print("rl run root:", RUN_ROOT)
print("rl output:", OUTPUT_DIR)
print("model:", MODEL_ID)


In [ ]:
# 3) Data split and Pilot C multitask record generation
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def compact_order(order):
    return " ".join(str(int(value)) for value in order)


def parse_compact_order(text, expected_len=4):
    values = [int(x) for x in re.findall(r"[1-4]", str(text))]
    if len(values) != expected_len or len(set(values)) != expected_len:
        return None
    return values


def row_image_paths(row, image_root=TRAIN_IMAGE_DIR):
    sample_id = str(row["Id"])
    return [str(Path(image_root) / sample_id / str(row[f"Input_{i}"])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def base_record(row_index, row):
    answer = [int(value) for value in row["Answer_list"]]
    order = order_to_sequence(answer)
    return {
        "row_index": int(row_index),
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order,
        "image_paths": row_image_paths(row, TRAIN_IMAGE_DIR),
    }


def pair_target_for_order(order, a, b):
    ranks = {frame: idx for idx, frame in enumerate(order)}
    return "A" if ranks[int(a)] < ranks[int(b)] else "B"


def build_pair_groups(base):
    groups = []
    order = base["order"]
    for i, j in PAIR_INDICES:
        a, b = i + 1, j + 1
        group = []
        for left, right in [(a, b), (b, a)]:
            item = copy.deepcopy(base)
            item.update({
                "task_type": "pairwise",
                "pair": [left, right],
                "image_paths": [base["image_paths"][left - 1], base["image_paths"][right - 1]],
                "target": pair_target_for_order(order, left, right),
            })
            group.append(item)
        groups.append(group)
    return groups


def build_record_pools(dataframe):
    pools = {task: [] for task in TASK_RATIOS}
    pools["pairwise_groups"] = []
    for row_index, row in dataframe.iterrows():
        base = base_record(row_index, row)
        order = base["order"]

        item = copy.deepcopy(base)
        item.update({"task_type": "order", "target": compact_order(order)})
        pools["order"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "first", "target": str(order[0])})
        pools["first"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "last", "target": str(order[-1])})
        pools["last"].append(item)

        pair_groups = build_pair_groups(base)
        pools["pairwise_groups"].extend(pair_groups)
        for group in pair_groups:
            pools["pairwise"].extend(group)

        first = order[0]
        remaining = [x for x in order if x != first]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_first", "fixed_first": first, "target": compact_order(remaining)})
        pools["fixed_first"].append(item)

        last = order[-1]
        remaining = [x for x in order if x != last]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_last", "fixed_last": last, "target": compact_order(remaining)})
        pools["fixed_last"].append(item)

        middle = [x for x in order if x not in {order[0], order[-1]}]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_endpoints", "fixed_first": order[0], "fixed_last": order[-1], "target": compact_order(middle)})
        pools["fixed_endpoints"].append(item)
    return pools


def sample_records(records, count, rng):
    records = list(records)
    if count <= len(records):
        indices = rng.choice(len(records), size=count, replace=False)
    else:
        base_indices = np.arange(len(records))
        extra_indices = rng.choice(len(records), size=count - len(records), replace=True)
        indices = np.concatenate([base_indices, extra_indices])
        rng.shuffle(indices)
    return [records[int(index)] for index in indices]


def sample_pair_groups(groups, target_record_count, rng):
    groups = list(groups)
    group_count = max(1, int(math.ceil(target_record_count / 2)))
    selected_groups = sample_records(groups, group_count, rng)
    records = [record for group in selected_groups for record in group]
    return records[:target_record_count] if len(records) > target_record_count else records


def build_balanced_records(dataframe):
    pools = build_record_pools(dataframe)
    base_total = int(math.ceil(len(pools["order"]) / TASK_RATIOS["order"]))
    rng = np.random.default_rng(SEED)
    merged = []
    distribution = {}
    for task, ratio in TASK_RATIOS.items():
        count = max(1, int(round(base_total * ratio)))
        if task == "pairwise":
            records = sample_pair_groups(pools["pairwise_groups"], count, rng)
            distribution[task] = {
                "pair_group_pool": len(pools["pairwise_groups"]),
                "record_pool": len(pools["pairwise"]),
                "sampled": len(records),
                "sampled_groups": int(math.ceil(len(records) / 2)),
            }
        else:
            records = sample_records(pools[task], count, rng)
            distribution[task] = {"pool": len(pools[task]), "sampled": len(records)}
        merged.extend(records)
        print(task, distribution[task])
    rng.shuffle(merged)
    return merged, pools, distribution

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["Id"] = train_df["Id"].astype(str)
test_df["Id"] = test_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

split_root = SNU_ROOT / "id_splits" / "qwen2vl_lgt_order_refine_20260714_003635"
if (split_root / "train_ids.json").exists() and (split_root / "validation_ids.json").exists():
    train_ids = set(str(x) for x in json.load(open(split_root / "train_ids.json", "r", encoding="utf-8")))
    valid_ids = set(str(x) for x in json.load(open(split_root / "validation_ids.json", "r", encoding="utf-8")))
else:
    unique_ids = train_df["Id"].unique().copy()
    rng = np.random.default_rng(SEED)
    rng.shuffle(unique_ids)
    valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
    valid_ids = set(unique_ids[:valid_size])
    train_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(train_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)
if TRAIN_ROWS is not None:
    training_df = training_df.sample(n=min(TRAIN_ROWS, len(training_df)), random_state=SEED).reset_index(drop=True)
if VALID_ROWS is not None:
    validation_df = validation_df.sample(n=min(VALID_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)

train_records, train_pools, task_distribution = build_balanced_records(training_df)
valid_pools = build_record_pools(validation_df)
quick_eval_rows = int(globals().get("QUICK_EVAL_ROWS", 50))
quick_eval_df = validation_df.sample(n=min(quick_eval_rows, len(validation_df)), random_state=SEED).reset_index(drop=True)

run_config = {
    "experiment": "checkpoint1200_onpolicy_rl",
    "base_run_id": BASE_RUN_ID,
    "base_checkpoint_name": BASE_CHECKPOINT_NAME,
    "base_checkpoint_dir": str(BASE_CHECKPOINT_DIR),
    "rl_run_id": RUN_ID,
    "output_dir": str(OUTPUT_DIR),
    "task_ratios": TASK_RATIOS,
    "task_loss_weights": TASK_LOSS_WEIGHTS,
    "task_distribution": task_distribution,
    "learning_rate": LEARNING_RATE,
    "rl_steps": RL_STEPS,
    "rollouts_per_sample": ROLLOUTS_PER_SAMPLE,
    "temperature": TEMPERATURE,
    "sft_replay_weight": SFT_REPLAY_WEIGHT,
    "save_steps": SAVE_STEPS,
    "eval_steps": EVAL_STEPS,
    "seed": SEED,
    "train_rows": len(training_df),
    "validation_rows": len(validation_df),
}
with open(RUN_ROOT / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)
with open(OUTPUT_DIR / "task_distribution_rl.json", "w", encoding="utf-8") as f:
    json.dump(task_distribution, f, ensure_ascii=False, indent=2)

print("train/valid/test:", len(training_df), len(validation_df), len(test_df))
print("train records:", len(train_records))


In [ ]:
# 4) Prompt builders, dataset, collator
def task_instruction(example):
    sentence = example["sentence"]
    task_type = example["task_type"]
    if task_type == "pairwise":
        return (
            f"Caption:\n{sentence}\n\n"
            "The two candidate images are labeled A and B in the presented order.\n"
            "Which image occurs earlier in the story timeline?\n"
            "Answer only A or B."
        )
    if task_type == "first":
        return f"Caption:\n{sentence}\n\nWhich image is the first scene in the story? Answer only one frame number from 1 to 4."
    if task_type == "last":
        return f"Caption:\n{sentence}\n\nWhich image is the last scene in the story? Answer only one frame number from 1 to 4."
    if task_type == "order":
        return (
            f"Caption:\n{sentence}\n\n"
            "Order all four images from earliest to latest in the story.\n"
            "Answer only four frame numbers separated by spaces, for example: 1 2 3 4."
        )
    if task_type == "fixed_first":
        return (
            f"Caption:\n{sentence}\n\n"
            f"Frame {example['fixed_first']} is fixed as the first scene.\n"
            "Order the remaining frames from earliest to latest.\n"
            "Answer only the remaining frame numbers separated by spaces."
        )
    if task_type == "fixed_last":
        return (
            f"Caption:\n{sentence}\n\n"
            f"Frame {example['fixed_last']} is fixed as the last scene.\n"
            "Order the remaining frames from earliest to latest.\n"
            "Answer only the remaining frame numbers separated by spaces."
        )
    if task_type == "fixed_endpoints":
        return (
            f"Caption:\n{sentence}\n\n"
            f"Frame {example['fixed_first']} is fixed as the first scene.\n"
            f"Frame {example['fixed_last']} is fixed as the last scene.\n"
            "Order the remaining middle frames from earliest to latest.\n"
            "Answer only the remaining frame numbers separated by spaces."
        )
    raise ValueError(task_type)


def make_messages(example, include_answer=False):
    content = []
    for idx, _ in enumerate(example["image_paths"], start=1):
        label = "A" if example["task_type"] == "pairwise" and idx == 1 else "B" if example["task_type"] == "pairwise" and idx == 2 else str(idx)
        content.append({"type": "text", "text": f"\nImage {label}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + task_instruction(example)})
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": str(example["target"])})
    return messages


class PilotCDataset(Dataset):
    def __init__(self, records):
        self.records = list(records)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return self.records[index]


class PilotCCollator:
    def __init__(self, processor):
        self.processor = processor
        self.tokenizer = processor.tokenizer
        self.assistant_prefix_ids = self.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def _mask_prompt(self, input_ids):
        ids = input_ids.tolist()
        labels = input_ids.clone()
        start = None
        prefix = self.assistant_prefix_ids
        for i in range(0, max(0, len(ids) - len(prefix) + 1)):
            if ids[i:i + len(prefix)] == prefix:
                start = i + len(prefix)
        if start is None:
            labels[:] = -100
        else:
            labels[:start] = -100
        labels[labels == self.tokenizer.pad_token_id] = -100
        return labels

    def __call__(self, batch):
        texts = []
        images = []
        task_types = []
        for example in batch:
            texts.append(self.processor.apply_chat_template(make_messages(example, include_answer=True), tokenize=False, add_generation_prompt=False))
            images.append([load_rgb(path) for path in example["image_paths"]])
            task_types.append(example["task_type"])
        encoded = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        encoded["labels"] = torch.stack([self._mask_prompt(row) for row in encoded["input_ids"]])
        encoded["task_type"] = task_types
        return encoded


In [ ]:
# 5) Load checkpoint-1200 trainable adapter and define memory-safe RL helpers
import torch.nn.functional as F
import bitsandbytes as bnb
from peft import PeftModel

# Free stale objects before loading the 7B base model.
for _name in ["trainer", "model", "base_model", "processor", "optimizer", "train_log_rows"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory before model load: {free / (1024 ** 3):.1f} GB free / {total / (1024 ** 3):.1f} GB total")

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

model = PeftModel.from_pretrained(base_model, BASE_CHECKPOINT_DIR, is_trainable=True)
model.config.use_cache = False
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# Keep gradient checkpointing active with model.train(), but disable stochastic dropout
# so rollout scoring and RL gradient rescoring use the same policy.
DISABLED_DROPOUT_MODULES = 0
for module in model.modules():
    if isinstance(module, torch.nn.Dropout):
        module.p = 0.0
        DISABLED_DROPOUT_MODULES += 1
print("disabled dropout modules:", DISABLED_DROPOUT_MODULES)

model.train()
model.print_trainable_parameters()

optimizer = bnb.optim.PagedAdamW8bit(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
)
collator = PilotCCollator(processor)
rng = np.random.default_rng(SEED)
train_log_rows = []


def model_device(active_model):
    return next(active_model.parameters()).device


def batch_to_device(inputs, active_model):
    return {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}


def row_to_base(row_index, row):
    return base_record(int(row_index), row)


def example_from_base_for_order(base):
    item = copy.deepcopy(base)
    item.update({"task_type": "order", "target": compact_order(base["order"])})
    return item


def order_logprob_single(active_model, example, order):
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    try:
        prompt = processor.apply_chat_template(
            make_messages(example, include_answer=False),
            tokenize=False,
            add_generation_prompt=True,
        )
        images = [load_rgb(path) for path in example["image_paths"]]
        prompt_inputs = processor(text=[prompt], images=[images], return_tensors="pt")
        prompt_len = int(prompt_inputs["attention_mask"][0].sum().item())
        text = prompt + compact_order(order)
        inputs = processor(text=[text], images=[images], return_tensors="pt")
        inputs = batch_to_device(inputs, active_model)
        outputs = active_model(**inputs)
        input_ids = inputs["input_ids"][0]
        attention_len = int(inputs["attention_mask"][0].sum().item())
        target_len = attention_len - prompt_len
        target_ids = input_ids[prompt_len:prompt_len + target_len]
        logits = outputs.logits[0, prompt_len - 1:prompt_len - 1 + target_len]
        log_probs = F.log_softmax(logits.float(), dim=-1)
        return log_probs.gather(1, target_ids[:, None]).squeeze(1).mean()
    finally:
        processor.tokenizer.padding_side = old_padding_side


@torch.no_grad()
def order_logprob_scores_no_grad(active_model, example, candidate_orders, batch_size=None):
    was_training = active_model.training
    active_model.eval()
    batch_size = int(batch_size or RL_CANDIDATE_BATCH_SIZE)
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    try:
        prompt = processor.apply_chat_template(
            make_messages(example, include_answer=False),
            tokenize=False,
            add_generation_prompt=True,
        )
        images = [load_rgb(path) for path in example["image_paths"]]
        prompt_inputs = processor(text=[prompt], images=[images], return_tensors="pt")
        prompt_len = int(prompt_inputs["attention_mask"][0].sum().item())
        scores = []
        orders = list(candidate_orders)
        for start_index in range(0, len(orders), batch_size):
            batch_orders = orders[start_index:start_index + batch_size]
            texts = [prompt + compact_order(order) for order in batch_orders]
            batch_images = [images for _ in batch_orders]
            inputs = processor(text=texts, images=batch_images, padding=True, return_tensors="pt")
            inputs = batch_to_device(inputs, active_model)
            outputs = active_model(**inputs)
            for row_index, order in enumerate(batch_orders):
                input_ids = inputs["input_ids"][row_index]
                attention_len = int(inputs["attention_mask"][row_index].sum().item())
                target_len = attention_len - prompt_len
                target_ids = input_ids[prompt_len:prompt_len + target_len]
                logits = outputs.logits[row_index, prompt_len - 1:prompt_len - 1 + target_len]
                log_probs = F.log_softmax(logits.float(), dim=-1)
                scores.append(log_probs.gather(1, target_ids[:, None]).squeeze(1).mean().float().cpu())
        return torch.stack(scores)
    finally:
        processor.tokenizer.padding_side = old_padding_side
        if was_training:
            active_model.train()


def reward_components(pred_order, gold_order):
    pred_order = [int(x) for x in pred_order]
    gold_order = [int(x) for x in gold_order]
    exact = float(pred_order == gold_order)
    first = float(pred_order[0] == gold_order[0])
    last = float(pred_order[-1] == gold_order[-1])
    gold_edges = {(gold_order[i], gold_order[i + 1]) for i in range(3)}
    pred_edges = {(pred_order[i], pred_order[i + 1]) for i in range(3)}
    adjacency = len(gold_edges & pred_edges) / 3.0
    pred_rank = {frame: idx for idx, frame in enumerate(pred_order)}
    gold_rank = {frame: idx for idx, frame in enumerate(gold_order)}
    pair_accuracy = float(np.mean([
        (pred_rank[a] < pred_rank[b]) == (gold_rank[a] < gold_rank[b])
        for a, b in itertools.combinations([1, 2, 3, 4], 2)
    ]))
    total = 0.50 * exact + 0.20 * first + 0.15 * last + 0.10 * adjacency + 0.05 * pair_accuracy
    return {
        "reward": float(total),
        "exact": exact,
        "first": first,
        "last": last,
        "adjacency": float(adjacency),
        "pair": pair_accuracy,
    }


def sample_rollout_indices(policy_probs, rollouts_per_sample):
    probs = policy_probs.detach().float().cpu().numpy()
    probs = probs / probs.sum()
    k = min(int(rollouts_per_sample), len(probs))
    return [int(x) for x in rng.choice(len(probs), size=k, replace=False, p=probs)]


def prepare_rollouts_no_grad(active_model, base):
    example = example_from_base_for_order(base)
    scores = order_logprob_scores_no_grad(active_model, example, PERMUTATIONS)
    policy_probs = torch.softmax(scores.float() / TEMPERATURE, dim=0)
    entropy = float(-(policy_probs * torch.log(policy_probs.clamp_min(1e-12))).sum().item())
    sampled_indices = sample_rollout_indices(policy_probs, ROLLOUTS_PER_SAMPLE)
    sampled_orders = [list(PERMUTATIONS[index]) for index in sampled_indices]
    components = [reward_components(order, base["order"]) for order in sampled_orders]
    rewards_np = np.array([c["reward"] for c in components], dtype=np.float32)
    reward_mean = float(rewards_np.mean())
    reward_std = float(rewards_np.std())
    advantages_np = (rewards_np - reward_mean) / (reward_std + 1e-6)
    advantages_np = np.clip(advantages_np, -2.0, 2.0)
    advantages_np = advantages_np - advantages_np.mean()
    advantage_std = float(advantages_np.std())
    logs = {
        "mean_total_reward": reward_mean,
        "mean_exact_reward": float(np.mean([c["exact"] for c in components])),
        "mean_first_reward": float(np.mean([c["first"] for c in components])),
        "mean_last_reward": float(np.mean([c["last"] for c in components])),
        "mean_adjacency_reward": float(np.mean([c["adjacency"] for c in components])),
        "mean_pair_reward": float(np.mean([c["pair"] for c in components])),
        "rollout_unique_count": float(len({tuple(order) for order in sampled_orders})),
        "policy_entropy": entropy,
        "reward_std": reward_std,
        "advantage_std": advantage_std,
        "top1_order": compact_order(PERMUTATIONS[int(torch.argmax(policy_probs).item())]),
    }
    return example, sampled_indices, sampled_orders, advantages_np, logs


def backward_rl_rollouts_one_by_one(active_model, example, sampled_indices, advantages_np):
    was_training = active_model.training
    # Keep train mode so Transformers actually uses gradient checkpointing.
    # Dropout is already disabled globally by setting module.p = 0.0.
    active_model.train()
    losses = []
    try:
        scale = 1.0 / max(1, len(sampled_indices))
        for index, advantage in zip(sampled_indices, advantages_np):
            if abs(float(advantage)) < 1e-12:
                continue
            logprob = order_logprob_single(active_model, example, PERMUTATIONS[int(index)])
            loss = -(float(advantage) * logprob / TEMPERATURE) * scale
            loss.backward()
            losses.append(float(loss.detach().cpu().item()))
            del logprob, loss
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        return {
            "train_rl_loss": float(np.sum(losses)) if losses else 0.0,
            "sampled_indices": " ".join(str(int(x)) for x in sampled_indices),
        }
    finally:
        if was_training:
            active_model.train()
        else:
            active_model.eval()


def compute_sft_replay_loss(active_model, records, batch_size=SFT_REPLAY_BATCH_SIZE):
    if not records:
        return torch.tensor(0.0, device=model_device(active_model)), {"train_sft_replay_loss": 0.0}
    was_training = active_model.training
    active_model.train()
    batch = [records[int(i)] for i in rng.integers(0, len(records), size=int(batch_size))]
    inputs = collator(batch)
    task_types = inputs.pop("task_type")
    inputs = batch_to_device(inputs, active_model)
    outputs = active_model(**inputs)
    logits = outputs.logits[..., :-1, :].contiguous().float()
    labels = inputs["labels"][..., 1:].contiguous()
    token_losses = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        labels.view(-1),
        reduction="none",
        ignore_index=-100,
    ).view(labels.shape)
    mask = labels.ne(-100)
    sample_losses = (token_losses * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
    weights = torch.tensor([TASK_LOSS_WEIGHTS.get(task, 1.0) for task in task_types], dtype=sample_losses.dtype, device=sample_losses.device)
    loss = (sample_losses * weights).mean()
    if not was_training:
        active_model.eval()
    return loss, {"train_sft_replay_loss": float(loss.detach().cpu().item())}


def build_hard_sample_indices():
    if HARD_SAMPLE_RATIO <= 0:
        print("Hard sampling disabled for RL pilot; using random train samples only.")
        return [], list(range(len(training_df)))
    print("No train-specific hard cache is available in this notebook; hard sampling disabled to avoid validation leakage.")
    return [], list(range(len(training_df)))


hard_indices, random_indices = build_hard_sample_indices()


def choose_training_row_index():
    if hard_indices and rng.random() < HARD_SAMPLE_RATIO:
        return int(rng.choice(hard_indices))
    return int(rng.choice(random_indices))


def save_rl_checkpoint(step):
    save_dir = CHECKPOINT_ROOT / f"checkpoint-1200-rl-step-{int(step)}"
    save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_dir)
    processor.save_pretrained(save_dir)
    torch.save({
        "step": int(step),
        "optimizer": optimizer.state_dict(),
        "rng_bit_generator_state": rng.bit_generator.state,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }, save_dir / "rl_state.pt")
    print("saved:", save_dir)
    return save_dir


def flush_train_log():
    log_path = OUTPUT_DIR / "rl_train_log.csv"
    pd.DataFrame(train_log_rows).to_csv(log_path, index=False)
    return log_path


In [ ]:
# 6) Run on-policy RL training and save checkpoints
for step in range(1, RL_STEPS + 1):
    row_index = choose_training_row_index()
    row = training_df.iloc[row_index]
    base = row_to_base(row_index, row)

    optimizer.zero_grad(set_to_none=True)
    example, sampled_indices, sampled_orders, advantages_np, rollout_logs = prepare_rollouts_no_grad(model, base)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    rl_logs = backward_rl_rollouts_one_by_one(model, example, sampled_indices, advantages_np)
    sft_loss, sft_logs = compute_sft_replay_loss(model, train_records, batch_size=SFT_REPLAY_BATCH_SIZE)
    scaled_sft_loss = SFT_REPLAY_WEIGHT * sft_loss
    if not torch.isfinite(scaled_sft_loss):
        raise RuntimeError(f"Non-finite SFT loss at step {step}: {scaled_sft_loss}")
    scaled_sft_loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
    if not torch.isfinite(torch.as_tensor(grad_norm)):
        raise RuntimeError(f"Non-finite grad norm at step {step}: {grad_norm}")
    optimizer.step()

    total_loss_value = float(rl_logs["train_rl_loss"] + float(scaled_sft_loss.detach().cpu().item()))
    log_row = {
        "step": step,
        "sample_id": str(row["Id"]),
        "train_total_loss": total_loss_value,
        "grad_norm": float(grad_norm.detach().cpu().item() if torch.is_tensor(grad_norm) else grad_norm),
        **rollout_logs,
        **rl_logs,
        **sft_logs,
    }
    train_log_rows.append(log_row)
    if step % LOGGING_STEPS == 0 or step == 1:
        print(log_row)
    flush_train_log()

    if step % SAVE_STEPS == 0:
        save_rl_checkpoint(step)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

final_dir = OUTPUT_DIR / "final_rl_adapter"
final_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(final_dir)
processor.save_pretrained(final_dir)
torch.save({
    "step": int(RL_STEPS),
    "optimizer": optimizer.state_dict(),
    "rng_bit_generator_state": rng.bit_generator.state,
    "torch_rng_state": torch.get_rng_state(),
    "cuda_rng_state_all": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
}, final_dir / "rl_state.pt")
flush_train_log()
print("final adapter saved:", final_dir)


## Next

Evaluate RL checkpoints with `eval_checkpoint1200_dualbranch_pair_priority_decoder.ipynb` by pointing `SELECTED_CHECKPOINTS` or adapter paths to the saved RL checkpoint directories.
